In [1]:
import os
from dotenv import load_dotenv
import numpy as np
import subprocess
load_dotenv(override=True)

True

### ML Flow

In [2]:
import mlflow
from scripts.ml_flow_utils import set_experiment

In [3]:
GCS_ML_FLOW = os.environ["GCS_ML_FLOW"]
DATABASE_URI_ML_FLOW = os.environ["DATABASE_URI_ML_FLOW"]

In [5]:
# psql '{DATABASE_URI_ML_FLOW}'
# DROP SCHEMA public CASCADE;
# DROP SCHEMA extensions CASCADE;
# CREATE SCHEMA public;
# CREATE SCHEMA extensions;

In [6]:
mlflow.set_tracking_uri(DATABASE_URI_ML_FLOW)

In [7]:
experiment_name = "Testing"
set_experiment(experiment_name, GCS_ML_FLOW)

In [8]:
with mlflow.start_run(run_name="Testing"):
    mlflow.log_text("Testing", "testing.txt")    

In [9]:
# mlflow ui --backend-store-uri DATABASE_URI_ML_FLOW

In [10]:
mlflow.__version__

'3.11.1'

### Scripts Package

In [2]:
def create_package(source_path = "./scripts", package_name = "rf_donut_model"):
    # [1] Copy scripts
    import shutil
    from pathlib import Path
    
    output_dir = Path("wheel")
    package_dir = output_dir / package_name    
    
    shutil.rmtree(output_dir, ignore_errors=True)
    package_dir.mkdir(parents=True, exist_ok=True)
    
    source_dir = Path(source_path)
    for py_file in source_dir.glob("*.py"):
        shutil.copy2(py_file, package_dir)    
    
    # [2] Create __init__.py
    from pathlib import Path
    
    package_dir = Path(f"wheel/{package_name}")
    package_dir.mkdir(parents=True, exist_ok=True)
    
    imports = []
    
    for file in sorted(package_dir.glob("*.py")):
        if file.name.startswith("__"):
            continue
    
        module_name = file.stem
        imports.append(f"from .{module_name} import *")
    
    (package_dir / "__init__.py").write_text(
        "\n".join(imports) + "\n"
    )

    # [3] Create setup.py
    from pathlib import Path
    from textwrap import dedent
    
    setup_py = Path("wheel/setup.py")
    setup_py.write_text(
        dedent(f"""
        from setuptools import setup, find_packages
    
        setup(
            name="{package_name}",
            version="0.1.0",
            packages=find_packages(),
        )
        """).strip() + "\n"
    )

    # [4] Build package
    import os
    import subprocess
    os.chdir("wheel")
    subprocess.run(
        ["python", "-m", "build", "--wheel"],
        check=True
    )
    os.chdir("..")

In [8]:
create_package()

* Creating isolated environment: venv+pip...
* Installing packages in isolated environment:
  - setuptools >= 40.8.0
* Getting build dependencies for wheel...


running egg_info
creating rf_donut_model.egg-info
writing rf_donut_model.egg-info/PKG-INFO
writing dependency_links to rf_donut_model.egg-info/dependency_links.txt
writing top-level names to rf_donut_model.egg-info/top_level.txt
writing manifest file 'rf_donut_model.egg-info/SOURCES.txt'
reading manifest file 'rf_donut_model.egg-info/SOURCES.txt'
writing manifest file 'rf_donut_model.egg-info/SOURCES.txt'


* Building wheel...


running bdist_wheel
running build
running build_py
creating build/lib/rf_donut_model
copying rf_donut_model/train_eval.py -> build/lib/rf_donut_model
copying rf_donut_model/ground_truth_process.py -> build/lib/rf_donut_model
copying rf_donut_model/image_process.py -> build/lib/rf_donut_model
copying rf_donut_model/model.py -> build/lib/rf_donut_model
copying rf_donut_model/__init__.py -> build/lib/rf_donut_model
copying rf_donut_model/save_load_model.py -> build/lib/rf_donut_model
copying rf_donut_model/data_loader.py -> build/lib/rf_donut_model
copying rf_donut_model/ml_flow_utils.py -> build/lib/rf_donut_model
copying rf_donut_model/tokenizer.py -> build/lib/rf_donut_model
copying rf_donut_model/seed.py -> build/lib/rf_donut_model
copying rf_donut_model/inference.py -> build/lib/rf_donut_model
installing to build/bdist.linux-x86_64/wheel
running install
running install_lib
creating build/bdist.linux-x86_64/wheel
creating build/bdist.linux-x86_64/wheel/rf_donut_model
copying build/lib

#### Testing Package

In [9]:
!pip uninstall -y rf_donut_model

Found existing installation: rf_donut_model 0.1.0
Uninstalling rf_donut_model-0.1.0:
  Successfully uninstalled rf_donut_model-0.1.0


In [10]:
!pip install wheel/dist/rf_donut_model-0.1.0-py3-none-any.whl

Processing ./wheel/dist/rf_donut_model-0.1.0-py3-none-any.whl

[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [24]:
# from rf_donut_model.model import BARTDecoder

#### GCS Package

In [11]:
GCS_PACKAGE = os.environ["GCS_PACKAGE"]

In [12]:
!gcloud storage cp wheel/dist/rf_donut_model-0.1.0-py3-none-any.whl {GCS_PACKAGE}/rf_donut_model-0.1.0-py3-none-any.whl

Copying file://wheel/dist/rf_donut_model-0.1.0-py3-none-any.whl to gs://rf-pytorch-bucket/packages/rf_donut_model-0.1.0-py3-none-any.whl
  Completed files 1/1 | 10.8kiB/10.8kiB                                        


### KFP

In [27]:
GCS_PIPELINE = os.environ["GCS_PIPELINE"]

In [36]:
result = subprocess.run(
    ["gcloud", "storage", "ls", f"{GCS_PIPELINE}/"],
    capture_output=True,
    text=True
)

In [37]:
result.returncode == 0

True

### Google Secret

In [5]:
import os
import subprocess

DATABASE_URI_ML_FLOW = os.environ["DATABASE_URI_ML_FLOW"]

In [13]:
# DATABASE_URI_ML_FLOW

In [43]:
# subprocess.run(
#     [
#         "gcloud",
#         "secrets",
#         "create",
#         "DATABASE_URI_ML_FLOW",
#         "--data-file=-",
#     ],
#     input=DATABASE_URI_ML_FLOW,
#     text=True,
#     check=True,
# )

In [10]:
# subprocess.run(
#     [
#         "gcloud",
#         "secrets",
#         "versions",
#         "add",
#         "DATABASE_URI_ML_FLOW",
#         "--data-file=-",
#     ],
#     input=DATABASE_URI_ML_FLOW,
#     text=True,
#     check=True,
# )

In [51]:
# !gcloud secrets list --format="table(name,createTime)"

In [52]:
# !gcloud secrets delete DATABASE_URI --quiet

In [9]:
# !gcloud secrets versions access latest --secret=DATABASE_URI_ML_FLOW

In [49]:
# !gcloud auth list

### Library Version

In [57]:
import torch
import transformers
import timm
import mlflow
import numpy
import PIL
import torchvision
import datasets
from google.cloud import storage
from google.cloud import secretmanager
import psycopg2

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("timm:", timm.__version__)
print("mlflow:", mlflow.__version__)
print("numpy:", numpy.__version__)
print("Pillow:", PIL.__version__)
print("torchvision:", torchvision.__version__)
print("datasets:", datasets.__version__)
print("google-cloud-storage:", storage.__version__)
print("google-cloud-secret-manager:", secretmanager.__version__)
print("psycopg2:", psycopg2.__version__)

torch: 2.13.0+cu130
transformers: 5.14.1
timm: 1.0.26
mlflow: 3.11.1
numpy: 2.5.3
Pillow: 11.3.0
torchvision: 0.28.0+cu130
datasets: 5.0.0
google-cloud-storage: 3.13.0
google-cloud-secret-manager: 2.30.0
psycopg2: 2.9.9 (dt dec pq3 ext lo64)


In [8]:
import psycopg
psycopg.__version__

'3.1.19'

In [13]:
!python -m pip show psycopg

Name: psycopg
Version: 3.1.19
Summary: PostgreSQL database adapter for Python
Home-page: https://psycopg.org/psycopg3/
Author: Daniele Varrazzo
Author-email: daniele.varrazzo@gmail.com
License: GNU Lesser General Public License v3 (LGPLv3)
Location: /home/ridwanfatur/miniconda3/envs/py3_12_9/lib/python3.12/site-packages
Requires: typing-extensions
Required-by: langgraph-checkpoint-postgres, syft


### Dataset

In [54]:
from datasets import load_dataset
dataset_name = "ridwanFatur98/synth-invoice"

In [55]:
dataset = load_dataset(dataset_name)

In [56]:
dataset

DatasetDict({
    train: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 900
    })
    test: Dataset({
        features: ['image', 'ground_truth'],
        num_rows: 100
    })
})

### Docker Image

In [8]:
# !gcloud artifacts repositories create pytorch-repo \
#   --repository-format=docker \
#   --location=us-central1 \
#   --description="PyTorch Docker images"

In [9]:
def create_pytorch_dockerfile():
    from pathlib import Path

    docker_dir = Path("docker-file-pytorch")
    docker_dir.mkdir(exist_ok=True)

    dockerfile = """\
FROM pytorch/pytorch:2.13.0-cuda13.0-cudnn9-runtime

RUN printf "[global]\\nbreak-system-packages = true\\n" > /etc/pip.conf

RUN pip install --no-cache-dir \\
    --extra-index-url https://download.pytorch.org/whl/cu130 \\
    torchvision==0.28.0+cu130
"""

    dockerfile_path = docker_dir / "Dockerfile"
    dockerfile_path.write_text(dockerfile)

    print(f"Created: {dockerfile_path}")

In [10]:
create_pytorch_dockerfile()

Created: docker-file-pytorch/Dockerfile


In [11]:
PROJECT_ID = os.environ["PROJECT_ID"]

In [1]:
# !gcloud builds submit docker-file-pytorch --tag us-central1-docker.pkg.dev/{PROJECT_ID}/pytorch-repo/pytorch-sdk:latest